# 架构

Sohn、Lee、Yan，NeurIPS 2015。条件 `x` 和输出 `y` 进识别网络；生成时只给 `x`，从条件先验采样 `z`，再解码出 `y`。

```
x, y → q(z|x,y) → μ_q, log σ_q² → z = μ_q + ε σ_q
x    → p(z|x)   → μ_p, log σ_p²
z, x → p(y|x,z) → ŷ
```

训练目标是条件对数似然的变分下界：重构项减去 `KL(q(z|x,y) || p(z|x))`。这一格用展平的 28×28 和 10 维 one-hot，把三套 MLP 的维度走一遍。后面两格换成卷积，数据分别是 CIFAR-10 和 LFW。


In [1]:
# 架构：三套 MLP + 维度打印，不含训练
import torch
import torch.nn as nn
import torch.nn.functional as F


def reparameterize(mu, logvar):
    """z = mu + eps * sigma, eps ~ N(0, I)。网络输出 logvar，保证 sigma > 0。"""
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std


def kl_diag_gaussian(mu_q, logvar_q, mu_p, logvar_p):
    """KL( N(mu_q, σ_q²) || N(mu_p, σ_p²) )，对角协方差，对 batch 求和。"""
    logvar_q = logvar_q.clamp(-20.0, 20.0)
    logvar_p = logvar_p.clamp(-20.0, 20.0)
    var_q = logvar_q.exp()
    var_p = logvar_p.exp()
    kl = 0.5 * (logvar_p - logvar_q + (var_q + (mu_q - mu_p).pow(2)) / var_p - 1.0)
    return kl.sum()


# =====================================================================
# 1) 先验 p(z | x)：只看条件，输出高斯的均值和对数方差
# =====================================================================
class PriorNetwork(nn.Module):
    def __init__(self, x_dim: int = 10, hidden: int = 256, latent_dim: int = 20):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(x_dim, hidden),          # 10 -> 256
            nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden),         # 256 -> 256
            nn.ReLU(inplace=True),
        )
        self.fc_mu = nn.Linear(hidden, latent_dim)
        self.fc_logvar = nn.Linear(hidden, latent_dim)

    def forward(self, x):
        h = self.net(x)
        return self.fc_mu(h), self.fc_logvar(h)


# =====================================================================
# 2) 识别网络 q(z | x, y)：同时看条件和输出
# =====================================================================
class RecognitionNetwork(nn.Module):
    def __init__(self, y_dim: int = 784, x_dim: int = 10, hidden: int = 256, latent_dim: int = 20):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(y_dim + x_dim, hidden),  # [y; x] 794 -> 256
            nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden),         # 256 -> 256
            nn.ReLU(inplace=True),
        )
        self.fc_mu = nn.Linear(hidden, latent_dim)
        self.fc_logvar = nn.Linear(hidden, latent_dim)

    def forward(self, y, x):
        h = self.net(torch.cat([y, x], dim=1))
        return self.fc_mu(h), self.fc_logvar(h)


# =====================================================================
# 3) 生成网络 p(y | x, z)：[z; x] 还原成像素概率
# =====================================================================
class GenerationNetwork(nn.Module):
    def __init__(self, y_dim: int = 784, x_dim: int = 10, hidden: int = 256, latent_dim: int = 20):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim + x_dim, hidden),  # [z; x] 30 -> 256
            nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden),              # 256 -> 256
            nn.ReLU(inplace=True),
            nn.Linear(hidden, y_dim),               # 256 -> 784
            nn.Sigmoid(),
        )

    def forward(self, z, x):
        return self.net(torch.cat([z, x], dim=1))


class CVAE(nn.Module):
    def __init__(self, y_dim=784, x_dim=10, hidden=256, latent_dim=20):
        super().__init__()
        self.x_dim = x_dim
        self.latent_dim = latent_dim
        self.prior = PriorNetwork(x_dim, hidden, latent_dim)
        self.recognition = RecognitionNetwork(y_dim, x_dim, hidden, latent_dim)
        self.generation = GenerationNetwork(y_dim, x_dim, hidden, latent_dim)

    def forward(self, y, x):
        mu_q, logvar_q = self.recognition(y, x)
        mu_p, logvar_p = self.prior(x)
        z = reparameterize(mu_q, logvar_q)
        recon = self.generation(z, x)
        return recon, mu_q, logvar_q, mu_p, logvar_p


def cvae_loss(recon, y, mu_q, logvar_q, mu_p, logvar_p):
    """二值像素用 BCE；KL 对条件先验 p(z|x)。"""
    recon_nll = F.binary_cross_entropy(recon, y, reduction="sum")
    kl = kl_diag_gaussian(mu_q, logvar_q, mu_p, logvar_p)
    return recon_nll + kl, recon_nll, kl


# =====================================================================
# 4) 用 forward hook 打印每一层的输入/输出维度
# =====================================================================
def _fmt_shape(value):
    if isinstance(value, torch.Tensor):
        return str(tuple(value.shape))
    if isinstance(value, (tuple, list)) and value and isinstance(value[0], torch.Tensor):
        return " ".join(str(tuple(t.shape)) for t in value)
    return str(type(value).__name__)


@torch.no_grad()
def print_shapes(model, *inputs, name="Model"):
    print(f"\n{'=' * 72}\n>>> {name}\n{'=' * 72}")
    print(f"{'layer':<32}{'in shape':<22}{'out shape'}")
    print("-" * 72)
    handles = []

    def make_hook(idx):
        def hook(module, inp, out):
            cls = module.__class__.__name__
            if isinstance(module, nn.Linear):
                cls = f"Linear({module.in_features}->{module.out_features})"
            in_shape = tuple(inp[0].shape) if isinstance(inp, tuple) else tuple(inp.shape)
            print(f"[{idx:02d}] {cls:<26}{str(in_shape):<22}{_fmt_shape(out)}")
        return hook

    leaf_idx = 0
    for module in model.modules():
        if module is model or len(list(module.children())) > 0:
            continue
        handles.append(module.register_forward_hook(make_hook(leaf_idx)))
        leaf_idx += 1

    model.eval()
    joined = "  ".join(str(tuple(t.shape)) for t in inputs)
    print(f"{'Input':<32}{joined}")
    output = model(*inputs)
    for handle in handles:
        handle.remove()
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("-" * 72)
    print(f"Output: {_fmt_shape(output)}   |  trainable params: {n_params:,}")
    return output


if __name__ == "__main__":
    torch.manual_seed(0)
    Y_DIM, X_DIM, Z_DIM, BATCH = 784, 10, 20, 8
    y = torch.rand(BATCH, Y_DIM)
    x = F.one_hot(torch.randint(0, X_DIM, (BATCH,)), X_DIM).float()
    z = torch.randn(BATCH, Z_DIM)

    prior = PriorNetwork(X_DIM, 256, Z_DIM)
    recognition = RecognitionNetwork(Y_DIM, X_DIM, 256, Z_DIM)
    generation = GenerationNetwork(Y_DIM, X_DIM, 256, Z_DIM)
    print_shapes(prior, x, name="Prior  p(z | x)")
    print_shapes(recognition, y, x, name="Recognition  q(z | x, y)")
    print_shapes(generation, z, x, name="Generation  p(y | x, z)")

    model = CVAE(Y_DIM, X_DIM, 256, Z_DIM)
    recon, mu_q, logvar_q, mu_p, logvar_p = model(y, x)
    loss, recon_nll, kl = cvae_loss(recon, y, mu_q, logvar_q, mu_p, logvar_p)
    print(f"\nrecon {tuple(recon.shape)}")
    print(f"per-image  loss {loss.item() / BATCH:.2f}  recon {recon_nll.item() / BATCH:.2f}  kl {kl.item() / BATCH:.2f}")


>>> Prior  p(z | x)
layer                           in shape              out shape
------------------------------------------------------------------------
Input                           (8, 10)
[00] Linear(10->256)           (8, 10)               (8, 256)
[01] ReLU                      (8, 256)              (8, 256)
[02] Linear(256->256)          (8, 256)              (8, 256)
[03] ReLU                      (8, 256)              (8, 256)
[04] Linear(256->20)           (8, 256)              (8, 20)
[05] Linear(256->20)           (8, 256)              (8, 20)
------------------------------------------------------------------------
Output: (8, 20) (8, 20)   |  trainable params: 78,888

>>> Recognition  q(z | x, y)
layer                           in shape              out shape
------------------------------------------------------------------------
Input                           (8, 784)  (8, 10)
[00] Linear(794->256)          (8, 794)              (8, 256)
[01] ReLU                  

# 实战1

数据是 CIFAR-10：32×32 RGB，10 类，训练 50,000、测试 10,000。条件 `x` 是类别 one-hot，输出 `y` 是整张图。先验只看标签，用 MLP；识别网络把标签铺成额外通道，和 RGB 一起做卷积；生成网络把 `[z; x]` 反卷积回 `3×32×32`。重构项是 `0.5||y-μ||²`，KL 的参照分布是条件先验 `p(z|x)`。


In [ ]:
# 实战1：自包含，不依赖上面的架构 cell
import json
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from matplotlib.ticker import MaxNLocator
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image

# ---------------------------------------------------------------------
# 配置
# CIFAR-10 已放在 ./data/cifar-10-batches-py，不重新下载。
# ---------------------------------------------------------------------
DATA_DIR = Path("./data")
OUT_DIR = Path("./runs/cifar10_cvae")
X_DIM = 10
LATENT_DIM = 128
HIDDEN_DIM = 256
BATCH_SIZE = 128
EPOCHS = 20                 # 冒烟测试可改成 1
LR = 1e-3
SEED = 0
N_PER_CLASS = 8

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
print("device:", device)

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


def reparameterize(mu, logvar):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std


def kl_diag_gaussian(mu_q, logvar_q, mu_p, logvar_p):
    """KL(q || p)，对角高斯，对 batch 求和。"""
    logvar_q = logvar_q.clamp(-20.0, 20.0)
    logvar_p = logvar_p.clamp(-20.0, 20.0)
    var_q = logvar_q.exp()
    var_p = logvar_p.exp()
    kl = 0.5 * (logvar_p - logvar_q + (var_q + (mu_q - mu_p).pow(2)) / var_p - 1.0)
    return kl.sum()


# ---------------------------------------------------------------------
# 网络
# 先验：x (B, 10) -> mu, logvar (B, latent)
# 识别：y (B, 3, 32, 32) 与铺开的标签 -> 32-16-8-4-2
# 生成：[z; x] -> 256x2x2 -> 3x32x32，sigmoid 把均值限制在 [0, 1]
# ---------------------------------------------------------------------
class PriorNetwork(nn.Module):
    def __init__(self, x_dim, hidden_dim, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(x_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True),
        )
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        nn.init.zeros_(self.fc_logvar.weight)
        nn.init.zeros_(self.fc_logvar.bias)

    def forward(self, x):
        h = self.net(x)
        return self.fc_mu(h), self.fc_logvar(h)


class RecognitionNetwork(nn.Module):
    def __init__(self, x_dim, latent_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3 + x_dim, 32, 4, stride=2, padding=1),   # 32 -> 16
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),           # 16 -> 8
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),          # 8 -> 4
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),         # 4 -> 2
            nn.ReLU(inplace=True),
        )
        self.fc_mu = nn.Linear(256 * 2 * 2, latent_dim)
        self.fc_logvar = nn.Linear(256 * 2 * 2, latent_dim)
        nn.init.zeros_(self.fc_logvar.weight)
        nn.init.zeros_(self.fc_logvar.bias)

    def forward(self, y, x):
        b, _, h, w = y.shape
        x_map = x[:, :, None, None].expand(b, x.size(1), h, w)
        hidden = self.conv(torch.cat([y, x_map], dim=1)).reshape(b, -1)
        return self.fc_mu(hidden), self.fc_logvar(hidden)


class GenerationNetwork(nn.Module):
    def __init__(self, x_dim, latent_dim):
        super().__init__()
        self.fc = nn.Linear(latent_dim + x_dim, 256 * 2 * 2)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),  # 2 -> 4
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),   # 4 -> 8
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),    # 8 -> 16
            nn.ReLU(inplace=True),
        )
        self.out = nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1)  # 16 -> 32
        nn.init.zeros_(self.out.bias)
        nn.init.xavier_uniform_(self.out.weight, gain=0.1)

    def forward(self, z, x):
        h = self.fc(torch.cat([z, x], dim=1)).view(z.size(0), 256, 2, 2)
        return torch.sigmoid(self.out(self.deconv(h)))


class CVAE(nn.Module):
    def __init__(self, x_dim=10, hidden_dim=256, latent_dim=128):
        super().__init__()
        self.x_dim = x_dim
        self.latent_dim = latent_dim
        self.prior = PriorNetwork(x_dim, hidden_dim, latent_dim)
        self.recognition = RecognitionNetwork(x_dim, latent_dim)
        self.generation = GenerationNetwork(x_dim, latent_dim)

    def forward(self, y, x):
        mu_q, logvar_q = self.recognition(y, x)
        mu_p, logvar_p = self.prior(x)
        z = reparameterize(mu_q, logvar_q)
        recon = self.generation(z, x)
        return recon, mu_q, logvar_q, mu_p, logvar_p

    @torch.no_grad()
    def generate(self, x, num_samples=1):
        """从 p(z|x) 采样 z，再解码。同一类别、不同 z 得到不同的图。"""
        if x.size(0) == 1 and num_samples > 1:
            x = x.expand(num_samples, -1).contiguous()
        mu_p, logvar_p = self.prior(x)
        z = reparameterize(mu_p, logvar_p)
        return self.generation(z, x)


def cvae_loss(recon, y, mu_q, logvar_q, mu_p, logvar_p):
    """高斯解码器、方差固定为 I。常数项与参数无关，这里略去。"""
    recon_nll = 0.5 * F.mse_loss(recon, y, reduction="sum")
    kl = kl_diag_gaussian(mu_q, logvar_q, mu_p, logvar_p)
    return recon_nll + kl, recon_nll, kl


# ---------------------------------------------------------------------
# 数据
# ---------------------------------------------------------------------
extracted = DATA_DIR / "cifar-10-batches-py"
if not extracted.is_dir():
    raise FileNotFoundError(f"找不到 CIFAR-10: {extracted}")

transform = transforms.ToTensor()
train_set = datasets.CIFAR10(DATA_DIR, train=True, download=False, transform=transform)
test_set = datasets.CIFAR10(DATA_DIR, train=False, download=False, transform=transform)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
classes = train_set.classes
print(f"CIFAR-10  train {len(train_set)}   test {len(test_set)}   steps/epoch {len(train_loader)}")
print("classes:", ", ".join(classes))

preview, preview_labels = next(iter(train_loader))
grid = make_grid(preview[:64], nrow=8, padding=2, pad_value=1.0)
plt.figure(figsize=(5.2, 5.2))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.title("CIFAR-10 training samples")
plt.axis("off")
plt.show()
print("first labels:", [classes[i] for i in preview_labels[:8].tolist()])


def class_samples(model):
    rows = []
    for class_id in range(model.x_dim):
        cond = F.one_hot(torch.tensor([class_id], device=device), X_DIM).float()
        rows.append(model.generate(cond, num_samples=N_PER_CLASS))
    return torch.cat(rows, dim=0)


@torch.no_grad()
def reconstruction_pair(model, images, labels):
    images = images.to(device)
    cond = F.one_hot(labels, X_DIM).float().to(device)
    recon, _, _, _, _ = model(images, cond)
    return torch.cat([images, recon], dim=0).cpu()


# ---------------------------------------------------------------------
# 训练
# 每个样本抽 1 次 z。loss = 0.5||y-μ||² + KL(q || p(z|x))
# ---------------------------------------------------------------------
model = CVAE(X_DIM, HIDDEN_DIM, LATENT_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
n_params = sum(p.numel() for p in model.parameters())
print(f"params: {n_params:,}  latent_dim: {LATENT_DIM}")

OUT_DIR.mkdir(parents=True, exist_ok=True)
history = []
snapshots = {}
fixed_images, fixed_labels = next(iter(test_loader))
fixed_images, fixed_labels = fixed_images[:N_PER_CLASS], fixed_labels[:N_PER_CLASS]

for epoch in range(1, EPOCHS + 1):
    model.train()
    total = recon_sum = kl_sum = 0.0
    n_seen = 0
    for images, labels in train_loader:
        images = images.to(device)
        cond = F.one_hot(labels, X_DIM).float().to(device)
        recon, mu_q, logvar_q, mu_p, logvar_p = model(images, cond)
        loss, recon_nll, kl = cvae_loss(recon, images, mu_q, logvar_q, mu_p, logvar_p)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
        recon_sum += recon_nll.item()
        kl_sum += kl.item()
        n_seen += images.size(0)

    model.eval()
    samples = class_samples(model).cpu()
    recon_pair = reconstruction_pair(model, fixed_images, fixed_labels)
    snapshots[epoch] = {"sample": samples, "recon": recon_pair}
    save_image(samples, OUT_DIR / f"sample_epoch{epoch:02d}.png", nrow=N_PER_CLASS)
    save_image(recon_pair, OUT_DIR / f"recon_epoch{epoch:02d}.png", nrow=N_PER_CLASS)

    row = {"epoch": epoch, "loss": total / n_seen, "recon": recon_sum / n_seen, "kl": kl_sum / n_seen}
    history.append(row)
    print(f"epoch {epoch:02d}/{EPOCHS}  loss {row['loss']:.4f}  recon {row['recon']:.4f}  kl {row['kl']:.4f}")

(OUT_DIR / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
torch.save(
    {"model": model.state_dict(), "x_dim": X_DIM, "hidden_dim": HIDDEN_DIM, "latent_dim": LATENT_DIM, "classes": classes},
    OUT_DIR / "cvae_cifar.pt",
)
print("saved", OUT_DIR / "cvae_cifar.pt")


# ---------------------------------------------------------------------
# 绘图
# ---------------------------------------------------------------------
epochs = [row["epoch"] for row in history]
fig, ax = plt.subplots(figsize=(7.4, 4.4))
ax.plot(epochs, [row["loss"] for row in history], marker="o", ms=3.5, label="loss = 重构 + KL")
ax.plot(epochs, [row["recon"] for row in history], marker="o", ms=3.5, label="重构 0.5||y-mu||^2")
ax.plot(epochs, [row["kl"] for row in history], marker="o", ms=3.5, label="KL(q || p)")
ax.set_xlabel("Epoch")
ax.set_ylabel("每张图")
ax.set_title("CIFAR-10 CVAE 训练曲线")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "loss.png", dpi=140)
plt.show()

show_epochs = sorted({1, max(1, EPOCHS // 2), EPOCHS})
fig, axes = plt.subplots(len(show_epochs), 2, figsize=(8.6, 3.15 * len(show_epochs)), squeeze=False)
for row, epoch in enumerate(show_epochs):
    sample = make_grid(snapshots[epoch]["sample"], nrow=N_PER_CLASS, padding=2, pad_value=1.0)
    recon = make_grid(snapshots[epoch]["recon"], nrow=N_PER_CLASS, padding=2, pad_value=1.0)
    axes[row, 0].imshow(sample.permute(1, 2, 0).numpy())
    axes[row, 1].imshow(recon.permute(1, 2, 0).numpy())
    axes[row, 0].set_title(f"Epoch {epoch}  按类别生成（每行一类，{N_PER_CLASS} 张）")
    axes[row, 1].set_title(f"Epoch {epoch}  上排原图，下排重构")
    for col in range(2):
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])
fig.suptitle("训练过程", y=1.01)
fig.tight_layout()
fig.savefig(OUT_DIR / "progress.png", dpi=140, bbox_inches="tight")
plt.show()


# ---------------------------------------------------------------------
# 测试：测试集上每张图的重构项和 KL
# ---------------------------------------------------------------------
model.eval()
total = recon_sum = kl_sum = 0.0
n_seen = 0
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        cond = F.one_hot(labels, X_DIM).float().to(device)
        recon, mu_q, logvar_q, mu_p, logvar_p = model(images, cond)
        loss, recon_nll, kl = cvae_loss(recon, images, mu_q, logvar_q, mu_p, logvar_p)
        total += loss.item()
        recon_sum += recon_nll.item()
        kl_sum += kl.item()
        n_seen += images.size(0)
print(f"test images: {n_seen}")
print(f"test loss {total / n_seen:.4f}  recon {recon_sum / n_seen:.4f}  kl {kl_sum / n_seen:.4f}")


# 实战2

数据是 LFW 人脸，缩到 64×64。上半张脸是条件 `x`，下半张脸是输出 `y`。先验和识别网络都看图像，生成网络在 4×4 特征上把条件编码和 `z` 拼起来，再反卷积回整张脸，最后把真实的上半张贴回去。同一张上半脸配不同的 `z`，得到不同的下半脸。损失只算下半张脸。默认 20 个 epoch；要看清五官把 `EPOCHS` 改成 100。


In [ ]:
# 实战2：自包含，不依赖上面的架构 cell 和实战1
import json
import tarfile
import urllib.request
import zipfile
from glob import glob
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from matplotlib.ticker import MaxNLocator
from PIL import Image
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms

# ---------------------------------------------------------------------
# 配置
# 官方 LFW 包下不动时，用同一批人脸的 multifaces 子集（一个人多张照片）。
# 目录已经在 ./data 里时不会重新下载。
# ---------------------------------------------------------------------
DATA_DIR = Path("./data")
OUT_DIR = Path("./runs/lfw_cvae")
SIZE = 64
LATENT_DIM = 128
BATCH_SIZE = 64
EPOCHS = 20                 # 五官更清楚可改成 100；冒烟测试可改成 1
LR = 1e-3
SEED = 0
MAX_BATCHES = 0             # 大于 0 时每个 epoch 只跑这么多 batch
N_FACES = 6
N_SAMPLES = 5
LFW_ZIPS = (
    (
        "lfw_multifaces-ingestion.zip",
        "https://hf-mirror.com/datasets/vilsonrodrigues/lfw/resolve/main/lfw_multifaces-ingestion.zip",
    ),
    (
        "lfw_multifaces-retrieval.zip",
        "https://hf-mirror.com/datasets/vilsonrodrigues/lfw/resolve/main/lfw_multifaces-retrieval.zip",
    ),
)

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
print("device:", device)

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


def reparameterize(mu, logvar):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std


def kl_diag_gaussian(mu_q, logvar_q, mu_p, logvar_p):
    """KL(q || p)，对角高斯，每个样本一个标量。"""
    logvar_q = logvar_q.clamp(-20.0, 20.0)
    logvar_p = logvar_p.clamp(-20.0, 20.0)
    var_q = logvar_q.exp()
    var_p = logvar_p.exp()
    kl = 0.5 * (logvar_p - logvar_q + (var_q + (mu_q - mu_p).pow(2)) / var_p - 1.0)
    return kl.sum(dim=-1)


def face_mask(batch, device):
    """上半张为 1（条件 x），下半张为 0（输出 y）。"""
    mask = torch.zeros(batch, 1, SIZE, SIZE, device=device)
    mask[:, :, : SIZE // 2, :] = 1.0
    return mask


def compose(pred, image, mask):
    return pred * (1.0 - mask) + image * mask


def input_panel(image, mask):
    return image * mask + 0.5 * (1.0 - mask)


# ---------------------------------------------------------------------
# 网络
# 卷积塔：64 -> 32 -> 16 -> 8 -> 4
# 先验看上半脸和 mask（4 通道），识别网络再拼上完整的脸（7 通道）
# 生成：条件特征与 z 在 4x4 上拼接，反卷积回 64x64
# ---------------------------------------------------------------------
def _conv_tower(in_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, 32, 4, stride=2, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(32, 64, 4, stride=2, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(64, 128, 4, stride=2, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(128, 256, 4, stride=2, padding=1),
        nn.ReLU(inplace=True),
    )


class PriorNetwork(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.conv = _conv_tower(4)
        self.fc_mu = nn.Linear(256 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(256 * 4 * 4, latent_dim)
        nn.init.zeros_(self.fc_logvar.weight)
        nn.init.zeros_(self.fc_logvar.bias)

    def forward(self, observed, mask):
        hidden = self.conv(torch.cat([observed, mask], dim=1)).reshape(observed.size(0), -1)
        return self.fc_mu(hidden), self.fc_logvar(hidden)


class RecognitionNetwork(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.conv = _conv_tower(7)
        self.fc_mu = nn.Linear(256 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(256 * 4 * 4, latent_dim)
        nn.init.zeros_(self.fc_logvar.weight)
        nn.init.zeros_(self.fc_logvar.bias)

    def forward(self, image, observed, mask):
        hidden = self.conv(torch.cat([image, observed, mask], dim=1)).reshape(image.size(0), -1)
        return self.fc_mu(hidden), self.fc_logvar(hidden)


class GenerationNetwork(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.cond = _conv_tower(4)
        self.fc = nn.Linear(latent_dim, 256 * 4 * 4)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 4, stride=2, padding=1),  # 4 -> 8
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),  # 8 -> 16
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),   # 16 -> 32
            nn.ReLU(inplace=True),
        )
        self.out = nn.ConvTranspose2d(64, 3, 4, stride=2, padding=1)  # 32 -> 64
        nn.init.zeros_(self.out.bias)
        nn.init.xavier_uniform_(self.out.weight, gain=0.1)

    def forward(self, z, observed, mask):
        cond = self.cond(torch.cat([observed, mask], dim=1))
        latent = self.fc(z).view(z.size(0), 256, 4, 4)
        hidden = self.deconv(torch.cat([cond, latent], dim=1))
        return torch.sigmoid(self.out(hidden))


class CVAE(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.latent_dim = latent_dim
        self.prior = PriorNetwork(latent_dim)
        self.recognition = RecognitionNetwork(latent_dim)
        self.generation = GenerationNetwork(latent_dim)

    def forward(self, image, observed, mask):
        mu_q, logvar_q = self.recognition(image, observed, mask)
        mu_p, logvar_p = self.prior(observed, mask)
        z = reparameterize(mu_q, logvar_q)
        recon = self.generation(z, observed, mask)
        return recon, mu_q, logvar_q, mu_p, logvar_p

    @torch.no_grad()
    def complete(self, image, observed, mask, num_samples):
        """从 p(z|x) 采样，生成下半张脸，再把真实的上半张贴回去。"""
        mu, logvar = self.prior(observed, mask)
        batch = image.size(0)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn(num_samples, batch, self.latent_dim, device=image.device)
        z = (mu.unsqueeze(0) + eps * std.unsqueeze(0)).reshape(num_samples * batch, -1)
        obs = observed.unsqueeze(0).expand(num_samples, -1, -1, -1, -1).reshape(num_samples * batch, 3, SIZE, SIZE)
        m = mask.unsqueeze(0).expand(num_samples, -1, -1, -1, -1).reshape(num_samples * batch, 1, SIZE, SIZE)
        recon = self.generation(z, obs, m).view(num_samples, batch, 3, SIZE, SIZE)
        return compose(recon, image.unsqueeze(0), mask.unsqueeze(0))


def cvae_loss(recon, image, mask, mu_q, logvar_q, mu_p, logvar_p):
    """只在下半张脸上计算 0.5||y-μ||²，上半张是给定条件。"""
    output = 1.0 - mask
    recon_nll = 0.5 * ((recon - image).pow(2) * output).sum(dim=(1, 2, 3))
    kl = kl_diag_gaussian(mu_q, logvar_q, mu_p, logvar_p)
    per_image = recon_nll + kl
    return per_image.mean(), recon_nll.mean(), kl.mean()


# ---------------------------------------------------------------------
# 数据
# ---------------------------------------------------------------------
def _jpg_dirs(root):
    found = []
    for name in ("lfw_funneled", "lfw_multifaces-ingestion", "lfw_multifaces-retrieval"):
        path = root / name
        if glob(str(path / "**" / "*.jpg"), recursive=True):
            found.append(path)
    return found


def ensure_lfw(root: Path):
    root.mkdir(parents=True, exist_ok=True)
    archive = root / "lfw-funneled.tgz"
    funneled = root / "lfw_funneled"
    if archive.is_file() and not glob(str(funneled / "**" / "*.jpg"), recursive=True):
        print("解压", archive, flush=True)
        with tarfile.open(archive, "r:gz") as tar:
            tar.extractall(root)
    if not _jpg_dirs(root):
        zip_dir = root / "lfw_zips"
        zip_dir.mkdir(parents=True, exist_ok=True)
        for filename, url in LFW_ZIPS:
            archive_path = zip_dir / filename
            if not archive_path.is_file() or archive_path.stat().st_size < 1_000_000:
                print("下载", filename, flush=True)
                urllib.request.urlretrieve(url, archive_path)
            print("解压", filename, flush=True)
            with zipfile.ZipFile(archive_path) as zf:
                zf.extractall(root)
    dirs = _jpg_dirs(root)
    if not dirs:
        raise FileNotFoundError(f"没有找到 LFW jpg: {root}")
    return dirs


class LFWFaces(Dataset):
    def __init__(self, image_roots):
        self.paths = sorted({p for root in image_roots for p in glob(str(Path(root) / "**" / "*.jpg"), recursive=True)})
        self.transform = transforms.Compose([
            transforms.Resize(250),
            transforms.CenterCrop(160),
            transforms.Resize((SIZE, SIZE)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image = Image.open(self.paths[index]).convert("RGB")
        return self.transform(image)


image_roots = ensure_lfw(DATA_DIR)
dataset = LFWFaces(image_roots)
n_test = max(1, int(0.1 * len(dataset)))
n_train = len(dataset) - n_test
train_set, test_set = random_split(dataset, [n_train, n_test], generator=torch.Generator().manual_seed(SEED))
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"LFW  images {len(dataset)}  train {n_train}  test {n_test}  steps/epoch {len(train_loader)}")

preview = next(iter(train_loader))[:16]
fig, axes = plt.subplots(2, 8, figsize=(8.2, 2.4))
for ax, image in zip(axes.reshape(-1), preview):
    ax.imshow(image.permute(1, 2, 0).numpy())
    ax.axis("off")
fig.suptitle("LFW training samples")
fig.tight_layout()
plt.show()


def completion_panels(model, images):
    images = images.to(device)
    mask = face_mask(images.size(0), device)
    observed = images * mask
    samples = model.complete(images, observed, mask, N_SAMPLES).cpu()
    return input_panel(images, mask).cpu(), images.cpu(), samples


# ---------------------------------------------------------------------
# 训练
# loss = 下半脸 0.5||y-μ||² + KL(q(z|x,y) || p(z|x))
# ---------------------------------------------------------------------
model = CVAE(LATENT_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
n_params = sum(p.numel() for p in model.parameters())
print(f"params: {n_params:,}  latent_dim: {LATENT_DIM}")

OUT_DIR.mkdir(parents=True, exist_ok=True)
history = []
snapshots = {}
fixed_images = next(iter(test_loader))[:N_FACES]


def save_completion(epoch, panels, originals, samples):
    cols = 2 + N_SAMPLES
    fig, axes = plt.subplots(N_FACES, cols, figsize=(1.35 * cols, 1.45 * N_FACES), squeeze=False)
    for row in range(N_FACES):
        axes[row, 0].imshow(panels[row].permute(1, 2, 0).numpy(), interpolation="nearest")
        axes[row, 1].imshow(originals[row].permute(1, 2, 0).numpy(), interpolation="nearest")
        for col in range(N_SAMPLES):
            axes[row, col + 2].imshow(samples[col, row].permute(1, 2, 0).clamp(0, 1).numpy(), interpolation="nearest")
        for col in range(cols):
            axes[row, col].set_xticks([])
            axes[row, col].set_yticks([])
    axes[0, 0].set_title("上半脸", fontsize=9)
    axes[0, 1].set_title("原图", fontsize=9)
    for col in range(N_SAMPLES):
        axes[0, col + 2].set_title(f"样本 {col + 1}", fontsize=9)
    fig.suptitle(f"Epoch {epoch}  同一张上半脸的不同补全", y=1.02)
    fig.tight_layout()
    path = OUT_DIR / f"complete_epoch{epoch:02d}.png"
    fig.savefig(path, dpi=140, bbox_inches="tight")
    plt.close(fig)
    return path


for epoch in range(1, EPOCHS + 1):
    model.train()
    total = recon_sum = kl_sum = 0.0
    seen = 0
    for step, images in enumerate(train_loader, start=1):
        if MAX_BATCHES and step > MAX_BATCHES:
            break
        images = images.to(device)
        mask = face_mask(images.size(0), device)
        observed = images * mask
        recon, mu_q, logvar_q, mu_p, logvar_p = model(images, observed, mask)
        loss, recon_nll, kl = cvae_loss(recon, images, mask, mu_q, logvar_q, mu_p, logvar_p)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        batch = images.size(0)
        total += loss.item() * batch
        recon_sum += recon_nll.item() * batch
        kl_sum += kl.item() * batch
        seen += batch
        if step == 1 or step % 50 == 0:
            print(f"  epoch {epoch:02d} step {step:04d} | loss {loss.item():.2f}", flush=True)

    row = {"epoch": epoch, "loss": total / seen, "recon": recon_sum / seen, "kl": kl_sum / seen}
    history.append(row)
    model.eval()
    panels, originals, samples = completion_panels(model, fixed_images)
    snapshots[epoch] = (panels, originals, samples)
    save_completion(epoch, panels, originals, samples)
    print(f"epoch {epoch:02d}/{EPOCHS}  loss {row['loss']:.4f}  recon {row['recon']:.4f}  kl {row['kl']:.4f}")

(OUT_DIR / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
torch.save({"model": model.state_dict(), "latent_dim": LATENT_DIM}, OUT_DIR / "cvae_lfw.pt")
print("saved", OUT_DIR / "cvae_lfw.pt")


# ---------------------------------------------------------------------
# 绘图
# ---------------------------------------------------------------------
epochs = [row["epoch"] for row in history]
fig, ax = plt.subplots(figsize=(7.4, 4.4))
ax.plot(epochs, [row["loss"] for row in history], marker="o", ms=3.5, label="loss = 重构 + KL")
ax.plot(epochs, [row["recon"] for row in history], marker="o", ms=3.5, label="下半脸重构")
ax.plot(epochs, [row["kl"] for row in history], marker="o", ms=3.5, label="KL(q || p)")
ax.set_xlabel("Epoch")
ax.set_ylabel("每张脸")
ax.set_title("LFW CVAE 训练曲线")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
if len(epochs) == 1:
    ax.set_xlim(epochs[0] - 0.5, epochs[0] + 0.5)
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "loss.png", dpi=140)
plt.show()

show_epochs = sorted({1, max(1, EPOCHS // 2), EPOCHS})
fig, axes = plt.subplots(len(show_epochs), 1, figsize=(12.5, 3.35 * len(show_epochs)), squeeze=False)
for row, epoch in enumerate(show_epochs):
    path = OUT_DIR / f"complete_epoch{epoch:02d}.png"
    axes[row, 0].imshow(plt.imread(path))
    axes[row, 0].set_title(f"Epoch {epoch}  上半脸 / 原图 / 五个补全样本")
    axes[row, 0].set_xticks([])
    axes[row, 0].set_yticks([])
fig.suptitle("训练过程", y=1.01)
fig.tight_layout()
fig.savefig(OUT_DIR / "progress.png", dpi=140, bbox_inches="tight")
plt.show()


# ---------------------------------------------------------------------
# 测试：测试集上每张脸的下半脸重构和 KL
# ---------------------------------------------------------------------
model.eval()
total = recon_sum = kl_sum = 0.0
seen = 0
with torch.no_grad():
    for images in test_loader:
        images = images.to(device)
        mask = face_mask(images.size(0), device)
        observed = images * mask
        recon, mu_q, logvar_q, mu_p, logvar_p = model(images, observed, mask)
        loss, recon_nll, kl = cvae_loss(recon, images, mask, mu_q, logvar_q, mu_p, logvar_p)
        batch = images.size(0)
        total += loss.item() * batch
        recon_sum += recon_nll.item() * batch
        kl_sum += kl.item() * batch
        seen += batch
print(f"test images: {seen}")
print(f"test loss {total / seen:.4f}  recon {recon_sum / seen:.4f}  kl {kl_sum / seen:.4f}")
